In [6]:
import pandas as pd

# --- Load the 3 files that cover Jan 2012 – Dec 2016 ---
df2 = pd.read_csv("Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv")
df3 = pd.read_csv("Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv")
df4 = pd.read_csv("Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv")

# --- Filter df2 to only Jan 2012 and Feb 2012 ---
df2 = df2[df2["month"].isin(["2012-01", "2012-02"])].copy()
print(f"df2 filtered to 2012-01 and 2012-02: {len(df2):,} rows")

df2 filtered to 2012-01 and 2012-02: 3,188 rows


1. Combine the datasets into a single master dataset. The combined dataset should contain all attributes
found in all files.
I will combine the 3 csv files in the next cell:
Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv (filtered to only include 2012-01 and 2012-02)
Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv,
Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv 

In [7]:
# --- Combine in order ---
df = pd.concat([df2, df3, df4], ignore_index=True)

print(f"\nCombined shape: {df.shape}")
print(f"Month range: {df['month'].min()} to {df['month'].max()}")
df.head()


Combined shape: (92544, 11)
Month range: 2012-01 to 2016-12


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,NaN
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,NaN
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,NaN
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,NaN
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,NaN


This produces the first mandatory output group, raw. Nothing is done except to filter away data before 2026-01.

In [8]:
import os

os.makedirs("output/raw", exist_ok=True)

df.to_csv("output/raw/raw.csv", index=False)

print(f"✅ Raw exported: output/raw/raw.csv ({len(df):,} rows)")
print(f"   Columns: {list(df.columns)}")

✅ Raw exported: output/raw/raw.csv (92,544 rows)
   Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

2. Perform data profiling on the dataset. Candidates may code their own profiling rules or leverage opensource
data profiling frameworks.
In the next few cells, I am examining the dataset to understand its structure, content, and quality:
-How many count of rows and columns are there?
-What are the data types?
-What are the min, max, average?
-How many unique values are in each column?

In [10]:
print("=" * 60)
print("COLUMN-LEVEL PROFILING")
print("=" * 60)
#This creates a column-level profiling summary table, showing data type, distinct values etc
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "non_null": df.notnull().sum(),
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "unique_values": df.nunique(),
})

print(profile)

COLUMN-LEVEL PROFILING
                       dtype  non_null  null_count  null_pct  unique_values
month                 object     92544           0      0.00             60
town                  object     92544           0      0.00             26
flat_type             object     92544           0      0.00              7
block                 object     92544           0      0.00           2139
street_name           object     92544           0      0.00            522
storey_range          object     92544           0      0.00             25
floor_area_sqm       float64     92544           0      0.00            168
flat_model            object     92544           0      0.00             20
lease_commence_date    int64     92544           0      0.00             48
resale_price         float64     92544           0      0.00           2615
remaining_lease      float64     37153       55391     59.85             50


In [11]:
print("=" * 60)
print("NUMERIC COLUMN STATISTICS")
print("=" * 60)
#This creates numeric column statistics, showing the number of non-null values, average, spread, etc

df.describe()

NUMERIC COLUMN STATISTICS


,floor_area_sqm,lease_commence_date,resale_price,remaining_lease
count,92544.000000,92544.000000,9.254400e+04,37153.000000
mean,96.569115,1990.072701,4.509390e+05,73.913116
std,24.682292,10.446719,1.281813e+05,10.885456
min,31.000000,1966.000000,1.900000e+05,48.000000
25%,74.000000,1983.000000,3.570000e+05,66.000000
50%,95.000000,1988.000000,4.280000e+05,72.000000
75%,111.000000,1999.000000,5.150000e+05,83.000000
max,280.000000,2013.000000,1.150000e+06,97.000000


In [12]:
print("=" * 60)
print("CATEGORICAL COLUMN STATISTICS")
print("=" * 60)
#This creates categorical column statistics, showing the number of all, distinct values, etc

df.describe(include=["object"])

CATEGORICAL COLUMN STATISTICS


,month,town,flat_type,block,street_name,storey_range,flat_model
count,92544,92544,92544,92544,92544,92544,92544
unique,60,26,7,2139,522,25,20
top,2012-03,JURONG WEST,4 ROOM,2,YISHUN RING RD,04 TO 06,Model A
freq,2360,7573,36535,397,1629,21214,26447


In [13]:
print("=" * 60)
print("CATEGORY-LEVEL PROFILING")
print("=" * 60)
#This examines the distribution of values within the 4 main columns etc

for col in ["flat_type", "storey_range", "flat_model", "town"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())
    print(f"Unique values: {df[col].nunique()}")

CATEGORY-LEVEL PROFILING

--- flat_type ---
4 ROOM              36535
3 ROOM              26307
5 ROOM              21368
EXECUTIVE            7295
2 ROOM                956
1 ROOM                 56
MULTI-GENERATION       27
Name: flat_type, dtype: int64
Unique values: 7

--- storey_range ---
04 TO 06    21214
07 TO 09    18765
01 TO 03    17466
10 TO 12    16028
13 TO 15     6704
16 TO 18     2706
01 TO 05     2700
06 TO 10     2474
11 TO 15     1259
19 TO 21     1156
22 TO 24      746
25 TO 27      381
16 TO 20      265
28 TO 30      236
21 TO 25       92
34 TO 36       88
37 TO 39       81
31 TO 33       79
26 TO 30       39
40 TO 42       38
46 TO 48        8
43 TO 45        8
36 TO 40        7
31 TO 35        2
49 TO 51        2
Name: storey_range, dtype: int64
Unique values: 25

--- flat_model ---
Model A                   26447
Improved                  24117
New Generation            16495
Premium Apartment          8314
Simplified                 5152
Apartment               

In [14]:
import re
import json

# Load the 2000 - Feb 2012 file
df_source = pd.read_csv("Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv")

# Filter for Jan 2012 only — this is the authoritative set
authoritative_set = df_source[df_source["month"] == "2012-01"].copy()

print(f"Total rows in authoritative set: {len(authoritative_set)}")
print(f"Columns: {list(authoritative_set.columns)}")
authoritative_set.head()

Total rows in authoritative set: 1559
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
366463,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0
366464,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0
366465,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0
366466,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0
366467,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0


3. Design data validation rules to validate Date, Town, Flat Type, Flat Model, and storey_range using Jan 2012
dataset as the authoritative set.

In [15]:
print("=" * 60)
print("AUTHORITATIVE REFERENCE VALUES (from Jan 2012)")
print("=" * 60)
#This extracts the set of allowed values for each validated column from the 2012-01 set, then prints them in a list

valid_towns = sorted(authoritative_set["town"].dropna().unique())
valid_flat_types = sorted(authoritative_set["flat_type"].dropna().unique())
valid_flat_models = sorted(authoritative_set["flat_model"].dropna().unique())
valid_storey_ranges = sorted(authoritative_set["storey_range"].dropna().unique())

print(f"\n--- TOWN ({len(valid_towns)} unique values) ---")
for t in valid_towns:
    print(f"  • {t}")

print(f"\n--- FLAT_TYPE ({len(valid_flat_types)} unique values) ---")
for ft in valid_flat_types:
    print(f"  • {ft}")

print(f"\n--- FLAT_MODEL ({len(valid_flat_models)} unique values) ---")
for fm in valid_flat_models:
    print(f"  • {fm}")

print(f"\n--- STOREY_RANGE ({len(valid_storey_ranges)} unique values) ---")
for sr in valid_storey_ranges:
    print(f"  • {sr}")

AUTHORITATIVE REFERENCE VALUES (from Jan 2012)

--- TOWN (26 unique values) ---
  • ANG MO KIO
  • BEDOK
  • BISHAN
  • BUKIT BATOK
  • BUKIT MERAH
  • BUKIT PANJANG
  • BUKIT TIMAH
  • CENTRAL AREA
  • CHOA CHU KANG
  • CLEMENTI
  • GEYLANG
  • HOUGANG
  • JURONG EAST
  • JURONG WEST
  • KALLANG/WHAMPOA
  • MARINE PARADE
  • PASIR RIS
  • PUNGGOL
  • QUEENSTOWN
  • SEMBAWANG
  • SENGKANG
  • SERANGOON
  • TAMPINES
  • TOA PAYOH
  • WOODLANDS
  • YISHUN

--- FLAT_TYPE (7 unique values) ---
  • 1 ROOM
  • 2 ROOM
  • 3 ROOM
  • 4 ROOM
  • 5 ROOM
  • EXECUTIVE
  • MULTI-GENERATION

--- FLAT_MODEL (13 unique values) ---
  • Adjoined flat
  • Apartment
  • Improved
  • Maisonette
  • Model A
  • Model A-Maisonette
  • Model A2
  • Multi Generation
  • New Generation
  • Premium Apartment
  • Simplified
  • Standard
  • Terrace

--- STOREY_RANGE (12 unique values) ---
  • 01 TO 03
  • 04 TO 06
  • 07 TO 09
  • 10 TO 12
  • 13 TO 15
  • 16 TO 18
  • 19 TO 21
  • 22 TO 24
  • 25 TO 27
  • 28 T

In [16]:
reference_values = {
    "town": valid_towns,
    "flat_type": valid_flat_types,
    "flat_model": valid_flat_models,
    "storey_range": valid_storey_ranges,
}
#This saves the authoritative reference list to json

with open("reference_values_jan2012.json", "w") as f:
    json.dump(reference_values, f, indent=2)

print("Reference values saved to 'reference_values_jan2012.json'")

Reference values saved to 'reference_values_jan2012.json'


In [17]:
# It runs 3 types of checks: month-format, categorical values must be in the authoritative set, and flags any missing
def validate_dataset(df, reference, month_col="month"):
    """
    Validate a dataset against the authoritative reference values from Jan 2012.
    """
    print("=" * 60)
    print("DATA VALIDATION REPORT")
    print("=" * 60)
    
    issues = {}
    
    # 1. Validate Month format
    print("\n[1] MONTH FORMAT (YYYY-MM)")
    pattern = r"^\d{4}-(0[1-9]|1[0-2])$"
    invalid_months = df[~df[month_col].astype(str).str.match(pattern)]
    if len(invalid_months) == 0:
        print("  ✅ All month values follow YYYY-MM format")
    else:
        print(f"  ❌ {len(invalid_months)} invalid month values")
        print(f"     Examples: {invalid_months[month_col].unique()[:5]}")
        issues["month"] = invalid_months[month_col].unique().tolist()
    
    # 2. Validate categorical columns
    for col in ["town", "flat_type", "flat_model", "storey_range"]:
        print(f"\n[{col.upper()}]")
        invalid = df[~df[col].isin(reference[col])]
        if len(invalid) == 0:
            print(f"  ✅ All values are valid")
        else:
            print(f"  ❌ {len(invalid)} invalid values ({len(invalid)/len(df)*100:.2f}%)")
            print(f"     Invalid values found: {sorted(invalid[col].dropna().unique())}")
            issues[col] = sorted(invalid[col].dropna().unique().tolist())
    
    # 3. Check for nulls
    print("\n[NULL VALUES]")
    for col in ["month", "town", "flat_type", "flat_model", "storey_range"]:
        null_count = df[col].isnull().sum()
        if null_count == 0:
            print(f"  ✅ {col}: 0 nulls")
        else:
            print(f"  ⚠️ {col}: {null_count} nulls ({null_count/len(df)*100:.2f}%)")
    
    return issues

In [18]:
# Assuming 'df' is your combined dataset
issues = validate_dataset(df, reference_values)
#This returns the issues 

DATA VALIDATION REPORT

[1] MONTH FORMAT (YYYY-MM)
  ✅ All month values follow YYYY-MM format

[TOWN]
  ✅ All values are valid

[FLAT_TYPE]
  ✅ All values are valid

[FLAT_MODEL]
  ❌ 492 invalid values (0.53%)
     Invalid values found: ['2-room', 'DBSS', 'Improved-Maisonette', 'Premium Apartment Loft', 'Premium Maisonette', 'Type S1', 'Type S2']

[STOREY_RANGE]
  ❌ 6975 invalid values (7.54%)
     Invalid values found: ['01 TO 05', '06 TO 10', '11 TO 15', '16 TO 20', '21 TO 25', '26 TO 30', '31 TO 35', '36 TO 40', '37 TO 39', '40 TO 42', '43 TO 45', '46 TO 48', '49 TO 51']

[NULL VALUES]
  ✅ month: 0 nulls
  ✅ town: 0 nulls
  ✅ flat_type: 0 nulls
  ✅ flat_model: 0 nulls
  ✅ storey_range: 0 nulls


In [19]:
#It provides a detailed report of invalid values for each
def show_invalid_values(df, reference, columns):
    """
    Show all unique invalid values per column, with counts and year context.
    """
    print("=" * 70)
    print("INVALID VALUES REPORT (vs Authoritative Set — Jan 2012)")
    print("=" * 70)
    
    for col in columns:
        invalid_df = df[~df[col].isin(reference[col])]
        
        print(f"\n{'=' * 70}")
        print(f"COLUMN: {col.upper()}")
        print(f"{'=' * 70}")
        
        if len(invalid_df) == 0:
            print("  ✅ No invalid values")
            continue
        
        print(f"  Total invalid rows: {len(invalid_df)} ({len(invalid_df)/len(df)*100:.2f}%)")
        print(f"  Unique invalid values: {invalid_df[col].nunique()}")
        print()
        
        # Show each unique invalid value with count and year range
        summary = invalid_df.groupby(col).agg(
            count=(col, "size"),
            earliest_month=("month", "min"),
            latest_month=("month", "max"),
        ).sort_values("count", ascending=False)
        
        print(summary.to_string())

In [20]:
show_invalid_values(df, reference_values, ["town", "flat_type", "flat_model", "storey_range"])
#It produces the detailed invalid values reports

INVALID VALUES REPORT (vs Authoritative Set — Jan 2012)

COLUMN: TOWN
  ✅ No invalid values

COLUMN: FLAT_TYPE
  ✅ No invalid values

COLUMN: FLAT_MODEL
  Total invalid rows: 492 (0.53%)
  Unique invalid values: 7

                        count earliest_month latest_month
flat_model                                               
DBSS                      277        2014-03      2016-12
Type S1                   138        2014-12      2016-12
Type S2                    55        2015-01      2016-11
Improved-Maisonette        10        2012-07      2016-11
Premium Maisonette          6        2012-09      2016-04
Premium Apartment Loft      5        2015-11      2016-12
2-room                      1        2016-10      2016-10

COLUMN: STOREY_RANGE
  Total invalid rows: 6975 (7.54%)
  Unique invalid values: 13

              count earliest_month latest_month
storey_range                                   
01 TO 05       2700        2012-03      2012-05
06 TO 10       2474        2012-0

In [22]:
# Keep the original column from the source files
# (rename it so it's clearly distinguishable)
if "remaining_lease" in df.columns:
    df = df.rename(columns={"remaining_lease": "remaining_lease_original"})
    print(f"Preserved original column as 'remaining_lease_original'")
    print(f"Non-null values: {df['remaining_lease_original'].notna().sum()} / {len(df)}")
else:
    print("No original 'remaining_lease' column found.")
    df["remaining_lease_original"] = pd.NA

Preserved original column as 'remaining_lease_original'
Non-null values: 37153 / 92544


4. Assume HDB lease is 99 years old and compute the remaining lease accordingly, rounded down to Years
and Months.
Assumptions:
-Lease start date is assumed to be 1 January of `lease_commence_date`, since the dataset only provides the year.
-Reference date: September 2026 (current month at time of test).

In [23]:
# Reference month — today is 2026-09
reference_month = pd.Period("2026-09", freq="M")

# Lease start = Jan of lease_commence_date
df["lease_start"] = pd.PeriodIndex(
    df["lease_commence_date"].astype(str) + "-01", freq="M"
)

# Lease expiry = lease start + 99 years (1188 months)
df["lease_expiry"] = df["lease_start"] + (99 * 12)

# Remaining months
df["remaining_months_total"] = (df["lease_expiry"] - reference_month).apply(lambda x: x.n)

# Split into years and months
df["remaining_lease_years"]  = df["remaining_months_total"] // 12
df["remaining_lease_months"] = df["remaining_months_total"] % 12

# Formatted string — new computed column
df["remaining_lease_computed"] = (
    df["remaining_lease_years"].astype(str) + " years " +
    df["remaining_lease_months"].astype(str) + " months"
)

# Check
print(df[[
    "lease_commence_date",
    "remaining_lease_original",
    "remaining_lease_computed"
]].head(10))

   lease_commence_date  remaining_lease_original remaining_lease_computed
0                 1979                       NaN        51 years 4 months
1                 1978                       NaN        50 years 4 months
2                 1978                       NaN        50 years 4 months
3                 1986                       NaN        58 years 4 months
4                 1986                       NaN        58 years 4 months
5                 1980                       NaN        52 years 4 months
6                 1986                       NaN        58 years 4 months
7                 1976                       NaN        48 years 4 months
8                 1981                       NaN        53 years 4 months
9                 1979                       NaN        51 years 4 months


5. Assume the composite key for the dataset to be all the columns excluding resale price. Where duplicates
are found, compare and take the higher resale price record as the reference.
Assumptions:
-Do not include remaining_lease in the composite key, because there are many rows where it is null.
-Do not include remaining_lease_computed as it is computed, not in the source.
-Where duplicates exist, the record with the higher `resale_price` is kept as the reference.

In [24]:
# This is to find the columns to use in composite key, and to use only source columns. This is my interpretation to 
#exclude columns we derived.
# All derived / annotation columns — exclude from composite key
derived_columns = [
    "remaining_lease",              # if it still exists
    "remaining_lease_original",     # source column (from file)
    "remaining_lease_computed",     # your computed column
    "remaining_lease_years",
    "remaining_lease_months",
    "remaining_months_total",
    "lease_start",
    "lease_expiry",
    "town_status",
]

# Composite key = all columns except resale_price and derived columns
key_columns = [
    col for col in df.columns
    if col != "resale_price" and col not in derived_columns
]

print("Composite key columns:")
for col in key_columns:
    print(f"  • {col}")

Composite key columns:
  • month
  • town
  • flat_type
  • block
  • street_name
  • storey_range
  • floor_area_sqm
  • flat_model
  • lease_commence_date


This produces the second mandatory output group, cleaned. As the assignment lists the outputs in this order, I assume that the ordering suggests that cleaned output is produced early, before transformation. Cleaned is exported after the deduplication step below — this is the point at which the dataset has passed the core data quality requirements: validation against the Jan 2012 authoritative set, remaining lease computation, and composite-key deduplication.

In [25]:
# Deduplicate the dataset using the composite key
df_dedup = (
    df
    .sort_values("resale_price", ascending=False)
    .drop_duplicates(subset=key_columns, keep="first")
    .reset_index(drop=True)
)

print(f"Original rows:  {len(df)}")
print(f"After dedup:    {len(df_dedup)}")
print(f"Rows removed:   {len(df) - len(df_dedup)}")

# ---- Export Cleaned output ----
import os
os.makedirs("output/cleaned", exist_ok=True)

df_cleaned = df_dedup.copy()
df_cleaned.to_csv("output/cleaned/cleaned.csv", index=False)

print(f"\n✅ Cleaned exported: output/cleaned/cleaned.csv ({len(df_cleaned):,} rows)")

Original rows:  92544
After dedup:    90944
Rows removed:   1600

✅ Cleaned exported: output/cleaned/cleaned.csv (90,944 rows)


6. Identify potentially anomalous resale price using appropriate heuristics.
Method: Segmented IQR by `(town, flat_type)`, k = 1.5. flag values outside 1.5 × IQR within each `(town, flat_type)` group
-Rationale: Resale prices vary significantly by town and flat type. A global
  threshold would misclassify premium models (e.g., `TERRACE`, `MAISONETTE`)
  as anomalies because of the low floor_type.
-Groups with fewer than 10 records are skipped to ensure stable quartiles.
-IQR is built on quartiles, which are resistant to extreme values. A handful of
very expensive or very cheap records won't shift Q1 or Q3 much.

In [26]:
# Average price by town
town_avg = df.groupby("town")["resale_price"].mean().sort_values(ascending=False)
print("Average price by town:")
print(town_avg)

# Average price by flat model
model_avg = df.groupby("flat_model")["resale_price"].mean().sort_values(ascending=False)
print("\nAverage price by flat model:")
print(model_avg)

Average price by town:
town
BUKIT TIMAH        678815.895833
BISHAN             589275.114758
CENTRAL AREA       582175.998667
MARINE PARADE      550672.873437
BUKIT MERAH        546911.348535
QUEENSTOWN         514352.128269
PASIR RIS          502352.450407
SERANGOON          482731.425826
PUNGGOL            481930.395330
KALLANG/WHAMPOA    481477.929829
SENGKANG           478051.042700
TAMPINES           469555.154429
TOA PAYOH          457503.732259
CLEMENTI           451571.369556
HOUGANG            441419.387489
SEMBAWANG          439380.377269
BUKIT PANJANG      437386.162527
JURONG EAST        429129.882443
CHOA CHU KANG      428985.860682
JURONG WEST        427011.437312
BEDOK              421496.629715
WOODLANDS          420026.962125
GEYLANG            418032.779162
ANG MO KIO         416705.057894
BUKIT BATOK        410030.846011
YISHUN             380752.456102
Name: resale_price, dtype: float64

Average price by flat model:
flat_model
Type S2                   990570.03636

7. Include any additional data profiling, cleaning or validation rules deemed appropriate. The actual dataset
may not contain anomalous data, but candidates should demonstrate mechanisms to identify such
behaviours.

In [27]:
def flag_segmented_anomalies(df, group_cols, price_col="resale_price", k=1.5):
    """
    Flag anomalies using IQR within each group.
    """
    df = df.copy()
    df["price_anomaly_seg"] = False
    
    for name, group in df.groupby(group_cols):
        if len(group) < 10:   # skip tiny groups
            continue
        Q1 = group[price_col].quantile(0.25)
        Q3 = group[price_col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - k * IQR
        upper = Q3 + k * IQR
        mask = (group[price_col] < lower) | (group[price_col] > upper)
        df.loc[group.index, "price_anomaly_seg"] = mask
    
    return df

# Segment by town + flat_type (your intuition)
df = flag_segmented_anomalies(df, ["town", "flat_type"])

print(f"Anomalies (town + flat_type): {df['price_anomaly_seg'].sum()} ({df['price_anomaly_seg'].mean()*100:.2f}%)")

Anomalies (town + flat_type): 1357 (1.47%)


In [28]:
anomalies = df[df["price_anomaly_seg"]].sort_values("resale_price", ascending=False)
#It inspects the flagged anomalies for review
print(anomalies[[
    "month", "town", "flat_type", "flat_model", "storey_range",
    "floor_area_sqm", "remaining_lease_computed", "resale_price"
]].head(20).to_string())

         month             town  flat_type  flat_model storey_range  floor_area_sqm remaining_lease_computed  resale_price
91881  2016-12  KALLANG/WHAMPOA     3 ROOM     Terrace     01 TO 03           259.0        44 years 4 months     1150000.0
85402  2016-08  KALLANG/WHAMPOA     5 ROOM        DBSS     28 TO 30           119.0        83 years 4 months     1100000.0
51375  2014-10           BISHAN  EXECUTIVE  Maisonette     22 TO 24           150.0        59 years 4 months     1088888.0
58504  2015-03  KALLANG/WHAMPOA     3 ROOM     Terrace     01 TO 03           280.0        44 years 4 months     1060000.0
37238  2013-11           BISHAN  EXECUTIVE  Maisonette     19 TO 21           150.0        59 years 4 months     1050000.0
59356  2015-04           BISHAN  EXECUTIVE  Maisonette     22 TO 24           149.0        59 years 4 months     1050000.0
84603  2016-08           BISHAN  EXECUTIVE  Maisonette     07 TO 09           153.0        59 years 4 months     1050000.0
88153  2016-10  

9. Using the cleaned data from the previous step, create a new column called Resale Identifier derived from
the rules below.

This produces the third mandatory output group, transformed.
Transformed is derived from Cleaned by adding the `resale_identifier`
  column.

In [29]:
import os
import hashlib

# --- Transformed = Cleaned + resale_identifier ---
df_transformed = df_cleaned.copy()

# 1. First character: always "S"
# 2. Next 3 digits: first 3 numeric digits of block, zero-padded
df_transformed["block_numeric"] = (
    df_transformed["block"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)   # remove non-numeric characters
    .str.zfill(3)                          # pad to at least 3 digits
    .str[:3]                               # take first 3 digits
)

# 3. Next 2 digits: first 2 digits of average resale price by (month, town, flat_type)
avg_price = (
    df_transformed.groupby(["month", "town", "flat_type"])["resale_price"]
    .transform("mean")
)

df_transformed["avg_price_2digits"] = (
    avg_price.astype(int).astype(str).str[:2]
)

# 4. Last 2 digits: month of entry (e.g., "2012-01" → "01")
df_transformed["month_2digits"] = df_transformed["month"].str[-2:]

# 5. Final character: first character of town
df_transformed["town_initial"] = df_transformed["town"].str[0]

# Combine into the Resale Identifier
df_transformed["resale_identifier"] = (
    "S" +
    df_transformed["block_numeric"] +
    df_transformed["avg_price_2digits"] +
    df_transformed["month_2digits"] +
    df_transformed["town_initial"]
)

# --- Export Transformed ---
os.makedirs("output/transformed", exist_ok=True)
df_transformed.to_csv("output/transformed/transformed.csv", index=False)

print(f"✅ Transformed exported: output/transformed/transformed.csv ({len(df_transformed):,} rows)")
print(f"   Columns: {len(df_transformed.columns)}")
print(f"   Sample identifier: {df_transformed['resale_identifier'].iloc[0]}")

✅ Transformed exported: output/transformed/transformed.csv (90,944 rows)
   Columns: 23
   Sample identifier: S0573612K


10. Hash the identifier column using an irreversible hashing algorithm while preserving uniqueness.
I used SHA-256, a hashing algorithm. It is a mathematical function that takes any input and returns a 64-character hexadecimal string that looks random. It operates in rounds, 64. Each round mixes bits using bitwise operations, rotates bits by fixed amounts, adds modular arithmetic, and uses nonlinear functions. It is mathematically infeasible to reverse because after 64 rounds, the relationship between input and output is so scrambled that no bit of the output tells you anything about any bit of the input, and with 2^256 possible outputs, astronomically unlikely for 2 different inputs to produce the same output.

This produces the fourth mandatory output group, hashed.
Hashed is derived from Transformed by adding the hashed identifier column.

In [30]:
import hashlib
import os

# --- Hash the identifier using SHA-256 (irreversible) ---
def hash_identifier(identifier):
    """
    Hash an identifier using SHA-256.
    Returns the 64-character hexadecimal digest.
    """
    return hashlib.sha256(str(identifier).encode("utf-8")).hexdigest()

df_hashed = df_transformed.copy()
df_hashed["resale_identifier_hashed"] = (
    df_hashed["resale_identifier"].apply(hash_identifier)
)

# --- Verify uniqueness is preserved ---
original_unique = df_hashed["resale_identifier"].nunique()
hashed_unique   = df_hashed["resale_identifier_hashed"].nunique()

print(f"Unique identifiers:   {original_unique:,}")
print(f"Unique hashed values: {hashed_unique:,}")

if original_unique == hashed_unique:
    print("✅ Uniqueness preserved — no collisions detected.")
else:
    print(f"❌ Collision detected: {original_unique - hashed_unique} values lost.")

# --- Export Hashed output ---
os.makedirs("output/hashed", exist_ok=True)

df_hashed.to_csv("output/hashed/hashed.csv", index=False)

print(f"\n✅ Hashed exported: output/hashed/hashed.csv ({len(df_hashed):,} rows)")
print(f"   Columns: {len(df_hashed.columns)}")

Unique identifiers:   77,255
Unique hashed values: 77,255
✅ Uniqueness preserved — no collisions detected.

✅ Hashed exported: output/hashed/hashed.csv (90,944 rows)
   Columns: 24


This produces the fifth mandatory output group, quarantined.
Quarantined collects records that failed any requirement — duplicate records, and anomalous data.

In [33]:
import os

# ============================================================
# STEP: BUILD QUARANTINED OUTPUT (duplicates + anomalies)
# ============================================================

# --- Work on a single sorted DataFrame so all masks align ---
df_sorted = (
    df
    .sort_values("resale_price", ascending=False)
    .reset_index(drop=True)
)

# --- Mask 1: Duplicates ---
# Duplicates = rows that drop_duplicates(keep="first") would remove
duplicate_mask = df_sorted.duplicated(subset=key_columns, keep="first")

# --- Mask 2: Anomalies ---
# anomaly flag is already in df_sorted (carried over from the sort)
anomaly_mask = df_sorted["price_anomaly_seg"]

# --- Combine ---
quarantine_mask = duplicate_mask | anomaly_mask

# --- Build quarantined dataset ---
df_quarantined = df_sorted[quarantine_mask].copy()

# --- Reason column ---
def get_reason(idx):
    reasons = []
    if duplicate_mask.loc[idx]:
        reasons.append("duplicate")
    if anomaly_mask.loc[idx]:
        reasons.append("anomaly")
    return "; ".join(reasons)

df_quarantined["quarantine_reason"] = [
    get_reason(i) for i in df_quarantined.index
]

# --- Export ---
os.makedirs("output/quarantined", exist_ok=True)
df_quarantined.to_csv("output/quarantined/quarantined.csv", index=False)

print(f"✅ Quarantined exported: output/quarantined/quarantined.csv")
print(f"   Total rows: {len(df_quarantined):,}")
print(f"\nReason breakdown:")
print(df_quarantined["quarantine_reason"].value_counts())

✅ Quarantined exported: output/quarantined/quarantined.csv
   Total rows: 2,941

Reason breakdown:
duplicate             1584
anomaly               1341
duplicate; anomaly      16
Name: quarantine_reason, dtype: int64
